# Post-class (advanced): MLIP for Li-ion diffusion

**MS.49900 AI for Materials Science, Hands-on session 7 (post-class, self-study)**

| Part | Question |
|---|---|
| A | How fast does Li diffuse in Li₆PS₅Cl, and what is its activation energy? |
| B | How many DFT structures does fine-tuning need, and can other training settings fix the forces? |
| C *(optional)* | Can a committee of models flag its own errors? |

This notebook is for self-study; nothing is submitted. Run it on a T4 GPU: Part A takes about 16 min (three 10 ps MD runs), Part B about 7 min and Part C about 4 min. Cells marked `# You can change here` are blanks. Each part ends with an explanation of the results.

## Setup

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU. Runtime > Change runtime type > T4 GPU, then run again from the top.')

In [ ]:
# Install SevenNet and download the course repository with the Li6PS5Cl DFT data (about 1 min)
!pip install -q sevenn==0.13.0

import os
if not os.path.exists('MS49900-AI4M'):
    !git clone -q https://github.com/kwongibaek/MS49900-AI4M.git

DATA = 'MS49900-AI4M/Data'
print(sorted(f for f in os.listdir(DATA) if f.startswith(('lpsc_', 'sevennet_'))))

In [ ]:
import time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ase.io import read
from ase.units import bar
from sevenn.calculator import SevenNetCalculator

to_kbar = 1 / bar / 1000          # eV/Å³ -> kbar


def dft_labels(traj):
    """DFT energy (eV/atom), forces (eV/Å) and stress (kbar) stored in the data file."""
    E = np.array([a.get_potential_energy() / len(a) for a in traj])
    F = np.concatenate([a.get_forces().ravel() for a in traj])
    S = np.concatenate([a.get_stress().ravel() for a in traj]) * to_kbar
    return E, F, S


def predict(calc, traj):
    """MLIP energy, forces and stress for every structure, plus seconds per structure."""
    E, F, S = [], [], []
    warm = traj[0].copy(); warm.calc = calc; warm.get_forces()   # warm-up, not timed
    t0 = time.time()
    for atoms in traj:
        a = atoms.copy()          # copy keeps the DFT labels of the original
        a.calc = calc
        E.append(a.get_potential_energy() / len(a))
        F.append(a.get_forces().ravel())
        S.append(a.get_stress().ravel() * to_kbar)
    sec = (time.time() - t0) / len(traj)
    return np.array(E), np.concatenate(F), np.concatenate(S), sec


def free_gpu(calc=None):
    del calc
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

import sevenn.util as util
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.optimize import LBFGS
from ase.filters import FrechetCellFilter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part A. Li diffusion and activation energy

Run NVT MD at three temperatures, get the Li diffusion coefficient from the MSD (Einstein relation), and fit an Arrhenius law.

$$\mathrm{MSD}(t) = 6Dt, \qquad D(T) = D_0 \exp\left(-\frac{E_a}{k_B T}\right)$$

**Model.** By default the fine-tuned model from class is used: upload the `checkpoint_fine_tuned.pth` you downloaded at the end of the in-class notebook to the Colab files panel. If it is not there, a fine-tuned checkpoint provided in the course repository is used. The explanation below was written with the provided checkpoint; your own model may give a different volume and activation energy.

In [ ]:
import os
# Settings
MODEL = 'checkpoint_fine_tuned.pth'   # (optional) '7net-0' repeats Part A with a universal MLIP (slower, about 30 min)
TEMPS = [600, 900, 1200]              # K
MD_STEPS = 5000                       # 2 fs x 5000 = 10 ps per temperature
FIT_START = 2.0                       # ps, skip the start of each run

FT_PATH = MODEL
if MODEL.endswith('.pth') and not os.path.exists(MODEL):
    FT_PATH = f'{DATA}/lpsc_finetuned_checkpoint.pth'
    print('Using the provided fine-tuned checkpoint')
print('Model:', FT_PATH)

In [ ]:
def relax(atoms, calc, fmax=0.05, steps=1000):
    a = atoms.copy(); a.calc = calc
    opt = LBFGS(FrechetCellFilter(a, hydrostatic_strain=True), logfile=None)
    opt.run(fmax=fmax, steps=steps)
    return a

def run_nvt(atoms, calc, T, n_steps, dt_fs=2.0, interval=10, seed=0):
    a = atoms.copy(); a.calc = calc
    MaxwellBoltzmannDistribution(a, temperature_K=T, rng=np.random.default_rng(seed))
    Stationary(a)
    dyn = Langevin(a, dt_fs * units.fs, temperature_K=T, friction=0.01 / units.fs,
                   rng=np.random.default_rng(seed + 1))
    pos, temp = [], []                 # the observer also runs at step 0
    def record():
        pos.append(a.get_positions().copy()); temp.append(a.get_temperature())
    dyn.attach(record, interval=interval)
    dyn.run(n_steps)
    t = np.arange(len(pos)) * interval * dt_fs / 1000
    return t, np.array(pos), np.array(temp)

def msd_of(pos, atoms, element):
    m = atoms.get_masses()
    disp = pos - pos[0]
    disp -= (disp * m[None, :, None]).sum(1, keepdims=True) / m.sum()   # remove drift
    sym = np.array(atoms.get_chemical_symbols())
    return (disp[:, sym == element] ** 2).sum(-1).mean(1)

calc = SevenNetCalculator(FT_PATH)
start = relax(read(f'{DATA}/lpsc_test_md.extxyz.gz', 0), calc)
print('relaxed volume per unit cell:', round(start.get_volume() / 8, 1), 'Å³')

runs, paths = {}, {}
for T in TEMPS:
    t0 = time.time()
    t, pos, temp = run_nvt(start, calc, T, MD_STEPS)
    runs[T] = (t, msd_of(pos, start, 'Li'), msd_of(pos, start, 'S'))
    paths[T] = pos[::2]                                  # keep positions for the Li-path plot
    print(f'{T} K done in {time.time() - t0:.0f} s, mean T = {temp[len(temp)//5:].mean():.0f} K')
free_gpu(calc)

**Blank 1.** Get D from the slope of the Li MSD.

In [ ]:
kB = 8.617333e-5   # eV/K

def diffusion_coefficient(t, msd_li, t_start=FIT_START):
    fit = t >= t_start
# You can change here------------------------------------------------------------
    slope, intercept = ...   # linear fit of MSD vs time in the fit window (Å²/ps)
    D = ...                  # Einstein relation in 3D, then Å²/ps -> cm²/s
# -------------------------------------------------------------------------------
    return D, slope, intercept

D_list = {}
fig, ax = plt.subplots(figsize=(6, 4))
for T, (t, msd_li, msd_s) in runs.items():
    D, slope, intercept = diffusion_coefficient(t, msd_li)
    D_list[T] = D
    line, = ax.plot(t, msd_li, label=f'Li {T} K')
    ax.plot(t, msd_s, ls=':', color=line.get_color())
    ax.plot(t[t >= FIT_START], slope * t[t >= FIT_START] + intercept, 'k--', lw=0.8)
ax.set_xlabel('Time (ps)'); ax.set_ylabel('MSD (Å²)'); ax.set_title('Li (solid), S (dotted)')
ax.legend(); plt.tight_layout(); plt.show()

pd.Series(D_list, name='D_Li (cm²/s)').to_frame()

### Where do the Li ions go?

Top view of the bottom half of the cell: S, P and Cl at the start, and the path of every Li ion during the run.

In [ ]:
from ase.data import atomic_numbers
from ase.data.colors import jmol_colors

ATOM_COLORS = {el: jmol_colors[atomic_numbers[el]] for el in ['Li', 'P', 'S', 'Cl']}

def plot_li_paths(ax, atoms, pos, title):
    sym = np.array(atoms.get_chemical_symbols())
    sel = pos[0, :, 2] < atoms.cell[2, 2] / 2
    for el in ['S', 'P', 'Cl']:
        m = (sym == el) & sel
        ax.scatter(pos[0, m, 0], pos[0, m, 1], s=40, color=ATOM_COLORS[el], edgecolors='k', linewidths=0.3, label=el)
    for i in np.where((sym == 'Li') & sel)[0]:
        ax.plot(pos[:, i, 0], pos[:, i, 1], color='#A05EE0', lw=0.6)
    ax.plot([], [], color='#A05EE0', lw=2, label='Li path')
    ax.set_aspect('equal'); ax.set_xlabel('x (Å)'); ax.set_ylabel('y (Å)'); ax.set_title(title)

fig, axes = plt.subplots(1, len(paths), figsize=(5.5 * len(paths), 5.5), squeeze=False)
for ax, (T, p) in zip(axes[0], paths.items()):
    plot_li_paths(ax, start, p, f'{T} K, {MD_STEPS * 2 / 1000:g} ps')
axes[0, -1].legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()

**Blank 2.** Fit the Arrhenius law and get the activation energy.

In [ ]:
T_arr = np.array(list(D_list), dtype=float)
D_arr = np.array(list(D_list.values()))
# You can change here------------------------------------------------------------
slope, intercept = ...   # fit ln(D) vs 1/T
Ea = ...                 # activation energy (eV)
# -------------------------------------------------------------------------------

x = 1000 / T_arr
plt.figure(figsize=(5, 4))
plt.semilogy(x, D_arr, 'o', color='#2F5597')
xx = np.linspace(x.min() * 0.95, x.max() * 1.05, 50)
plt.semilogy(xx, np.exp(intercept + slope * xx / 1000), 'k--')
plt.xlabel('1000 / T (1/K)'); plt.ylabel('D_Li (cm²/s)')
plt.title(f'Arrhenius fit: Ea = {Ea:.3f} eV')
plt.tight_layout(); plt.show()
print(f'Ea = {Ea:.3f} eV')

**What the results show.**

- **Only Li diffuses.** After 10 ps the Li MSD reaches about 6, 18 and 36 Å² at 600, 900 and 1200 K, while the S MSD stays below 0.3 Å²: the framework only vibrates. At 600 K most Li ions move only within their local cage, and almost none end up more than 5 Å from where they started. At 1200 K more than half do, and the Li paths join into a network that runs through the cell.
- **Activation energy.** D rises from about 7 × 10⁻⁶ cm²/s at 600 K to 6 × 10⁻⁵ cm²/s at 1200 K, and the Arrhenius fit gives $E_a$ ≈ 0.23 eV. Reported experimental values for Li₆PS₅Cl depend on the sample and the method: mostly about 0.3–0.4 eV from impedance spectroscopy and about 0.2–0.3 eV for long-range Li motion from NMR (see the summary table in Buchberger *et al.*, *ACS Appl. Mater. Interfaces* **15**, 53417 (2023)). Our value is at the low end of this range.
- **Why it can differ from experiment.**
  1. The MD runs at 600–1200 K and the experiments near room temperature. Using one Arrhenius law for both ranges assumes that the diffusion mechanism does not change in between.
  2. The simulation cell is a perfect crystal. Impedance is measured on pellets, whose grain boundaries add resistance and can raise the measured $E_a$.
  3. With the provided checkpoint the relaxed cell (about 982 Å³ per unit cell) is about 2–3% larger than in experiment (956–960 Å³), which widens the paths and can lower the barrier. The model fine-tuned in class has its energy minimum at a smaller volume (near 963 Å³), so with your own checkpoint this effect is weaker.
  4. 10 ps and 416 atoms give few long jumps at 600 K, so D there is noisy, and three temperatures give only a rough fit.
  5. The model has force errors (see Part B).
- **(Optional) A universal MLIP.** With `MODEL = '7net-0'` the energies are off by 2.4 eV/atom, but the MD should still run normally: a constant energy offset does not change the forces, and only the forces drive the MD. Any difference in $E_a$ therefore comes from differences in the forces, that is, in the shape of the energy surface.

*Your numbers may differ slightly from run to run.*

## Part B. How much data does fine-tuning need?

Fine-tune `7net-0-small` with 8, 17, 42 and 80 structures from the 600 K and 1200 K data (same settings and 10 epochs as in class) and plot the test error against the number of training structures. The structures are taken in a random order, so the 42-structure model here is not exactly the one from class. As in class, the energy error is shown with and without the constant offset.

In [ ]:
from torch_geometric.loader import DataLoader
from sevenn.train.graph_dataset import SevenNetGraphDataset
from sevenn.train.trainer import Trainer
from sevenn.error_recorder import ErrorRecorder

# Settings
N_TRAIN_LIST = [8, 17, 42, 80]
EPOCHS = 10
N_TEST = 20

base_ckpt = f'{DATA}/sevennet_0_small_checkpoint.pth'
_, base_config = util.model_from_checkpoint(base_ckpt)
dataset = SevenNetGraphDataset(
    cutoff=base_config['cutoff'], root='sevenn_data', drop_info=False,
    files=[f'{DATA}/lpsc_600K.extxyz.gz', f'{DATA}/lpsc_1200K.extxyz.gz'],
)
pool = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(0))   # random order of the structures

test_all = read(f'{DATA}/lpsc_test_md.extxyz.gz', ':')
test = test_all[::len(test_all) // N_TEST][:N_TEST]
dft_E, dft_F, dft_S = dft_labels(test)


def test_errors(E, F, S):
    """Test-set RMSE: energy with and without the constant offset, forces and stress."""
    return {'Energy': rmse(dft_E, E), 'Energy (offset removed)': rmse(dft_E - dft_E.mean(), E - E.mean()),
            'Force': rmse(dft_F, F), 'Stress': rmse(dft_S, S)}


def finetune(n_train, out_path, epochs=EPOCHS, indices=None, **changes):
    """Fine-tune 7net-0-small on the first n_train structures of the pool (or on given indices).
    Keyword arguments change the training settings, e.g. force_loss_weight=1.0."""
    torch.manual_seed(0)
    model, config = util.model_from_checkpoint(base_ckpt)
    config.update({
        'device': device, 'is_ddp': False,
        'optimizer': 'adam', 'optim_param': {'lr': 0.004},
        'scheduler': 'linearlr',
        'scheduler_param': {'start_factor': 1.0, 'total_iters': epochs, 'end_factor': 0.0001},
        'loss': 'huber', 'loss_param': {'delta': 0.01},
        'force_loss_weight': 0.1, 'stress_loss_weight': 0.01,     # same settings as in class
    })
    config.update(changes)
    trainer = Trainer.from_config(model, config)
    rec = ErrorRecorder.from_config(config)
    train_idx = pool[:n_train] if indices is None else indices
    train_loader = DataLoader(dataset[train_idx], batch_size=4, shuffle=True)
    for _ in range(epochs):
        trainer.run_one_epoch(train_loader, is_train=True, error_recorder=rec)
        trainer.scheduler_step()
        rec.epoch_forward()
    trainer.write_checkpoint(out_path, config=config, epoch=epochs)
    free_gpu()
    return out_path

**Blank 3.** Fine-tune and evaluate for each training-set size.

In [ ]:
curve = {}
calc = SevenNetCalculator(base_ckpt)                 # 0 structures = pretrained only
E, F, S, _ = predict(calc, test); free_gpu(calc)
curve[0] = test_errors(E, F, S)

for n in N_TRAIN_LIST:
    t0 = time.time()
# You can change here------------------------------------------------------------
    # fine-tune with n structures, then predict the test set with the new checkpoint
    # (predict also returns the time per structure, which is not needed here)
    E, F, S, _ = ...
# -------------------------------------------------------------------------------
    curve[n] = test_errors(E, F, S)
    print(f'{n:3d} structures done in {time.time() - t0:.0f} s')

curve = pd.DataFrame(curve).T
curve.index.name = 'Training structures'
curve.round(4)

In [ ]:
units_ = {'Energy': 'eV/atom', 'Energy (offset removed)': 'eV/atom', 'Force': 'eV/Å', 'Stress': 'kbar'}
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
n = curve.index.values[1:]
for ax, col in zip(axes, curve.columns):
    ax.plot(n, curve[col].values[1:], 'o-', color='#2F5597', label='fine-tuned')
    ax.axhline(curve[col].iloc[0], color='k', ls='--', lw=1, label='pretrained (0 structures)')
    ax.set_xscale('log'); ax.set_xticks(n); ax.set_xticklabels([str(v) for v in n]); ax.minorticks_off()
    if col == 'Energy':
        ax.set_yscale('log')                          # the offset (2.4 eV/atom) is far above the rest
    else:
        ax.set_ylim(bottom=0)
    ax.set_xlabel('Training structures'); ax.set_ylabel(f'RMSE ({units_[col]})')
    ax.set_title(col)
axes[0].legend(fontsize=9)
plt.tight_layout(); plt.show()

### Can other training settings fix the forces?

In class, fine-tuning on 42 structures reduced the energy and stress errors but not the force error. Two possible causes were the small weight of the forces in the loss (0.1) and the large starting learning rate (0.004). Here we fine-tune on the same 42 structures with each of them changed (about 2 min).

In [ ]:
VARIANTS = {                                          # changes to the training settings of class
    'force weight 1.0': {'force_loss_weight': 1.0},
    'learning rate 0.001': {'optim_param': {'lr': 0.001}},
}
table = {'pretrained (no fine-tuning)': curve.loc[0], 'fine-tuned, class settings': curve.loc[42]}
for i, (name, changes) in enumerate(VARIANTS.items()):
    path = finetune(42, f'ft_42_variant{i}.pth', **changes)
    calc = SevenNetCalculator(path)
    E, F, S, _ = predict(calc, test)
    free_gpu(calc)
    table[f'fine-tuned, {name}'] = test_errors(E, F, S)
    print(name, 'done')
pd.DataFrame(table).T.round(4)

**What the results show.**

- **Energy offset.** With 8 structures (20 optimizer steps) only about half of the 2.4 eV/atom offset is removed, and with 17 structures about 0.06 eV/atom is still left. From 42 structures on it is gone: the energy RMSE with and without the offset is the same.
- **Energy without the offset** does not fall steadily: about 3, 17, 8.5 and 4 meV/atom with 8, 17, 42 and 80 structures, against 14 meV/atom for the pretrained model. Each point is one run with one random set of structures, so single points are noisy; only the trend from 17 to 80 structures is meaningful.
- **Stress** improves first: 8 structures already bring it from about 5.9 to 1.4 kbar, and more data changes it little.
- **Forces** need the most data. With the class settings the force error is not monotonic: 0.23, 0.35, 0.25 and 0.20 eV/Å with 8, 17, 42 and 80 structures, against 0.24 eV/Å before fine-tuning. Only with 80 structures is it clearly lower than the pretrained value.
- **Training settings.** With the same 42 structures, raising the force weight from 0.1 to 1.0 lowers every error: forces 0.25 → 0.16 eV/Å, stress 1.8 → 1.2 kbar, offset-removed energy 8.5 → 4.7 meV/atom. So the small force weight is likely a main reason why fine-tuning did not improve the forces in class. A smaller learning rate (0.001) makes everything worse: in 10 epochs it cannot even remove the energy offset (0.29 eV/atom is left), and the force error rises to 0.42 eV/Å, probably because the large energy error still dominates the loss. The starting learning rate of 0.004 is needed to move the energy reference quickly.
- **A fair comparison of data sizes.** With the same 10 epochs, 8 structures get 20 optimizer steps and 80 structures get 200, so part of the gain with more data is simply more training. A fairer test keeps the number of steps fixed, or trains each size until the validation error stops falling, and repeats with several random sets.
- **How many DFT structures?** For this electrolyte a few dozen structures fix the energy offset and the stress; the forces need about 80 structures or a larger force weight. One DFT MD snapshot of this cell costs hours of CPU time, so trying other training settings is much cheaper than computing more data.

*Your numbers may differ slightly from run to run.*

## Part C (optional). Can the models tell when they are unsure?

Fine-tune three models on different random sets of 42 structures (a **committee**). Where the three disagree, the prediction is less trustworthy. Check whether the committee spread follows the true force error.

This is the idea behind **active learning**: compute new DFT data only where the committee disagrees. About 4 min on a T4.

In [ ]:
N_COMMITTEE = 3
committee = []
for k in range(N_COMMITTEE):
    g = torch.Generator().manual_seed(100 + k)
    subset = pool[torch.randperm(len(pool), generator=g)[:42]]        # a different random set for each model
    committee.append(finetune(42, f'committee_{k}.pth', indices=subset))
    print(f'model {k} done')

Fs = []
for path in committee:
    calc = SevenNetCalculator(path)
    Fs.append(predict(calc, test)[1].reshape(-1, 3))
    free_gpu(calc)
Fs = np.array(Fs)                                          # models x atoms x 3
spread = Fs.std(axis=0).mean(axis=1)                       # committee disagreement per atom (eV/Å)
error = np.linalg.norm(Fs.mean(axis=0) - dft_F.reshape(-1, 3), axis=1)   # true force error per atom (eV/Å)
symbols = np.concatenate([a.get_chemical_symbols() for a in test])
r = np.corrcoef(spread, error)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
hb = axes[0].hexbin(spread, error, gridsize=40, bins='log', cmap='Blues', mincnt=1)
axes[0].set_xlabel('Committee spread (eV/Å)'); axes[0].set_ylabel('True force error (eV/Å)')
axes[0].set_title(f'Per atom, correlation r = {r:.2f}'); fig.colorbar(hb, ax=axes[0], label='atoms')
edges = np.quantile(spread, np.linspace(0, 1, 11))
idx = np.clip(np.digitize(spread, edges[1:-1]), 0, 9)
axes[1].plot([spread[idx == i].mean() for i in range(10)], [error[idx == i].mean() for i in range(10)], 'o-', color='#2F5597')
axes[1].set_xlabel('Committee spread, binned (eV/Å)'); axes[1].set_ylabel('Mean true error (eV/Å)')
axes[1].set_title('Larger spread, larger error?')
plt.tight_layout(); plt.show()

pd.DataFrame({'spread': spread, 'error': error, 'element': symbols}).groupby('element').mean().round(3)

**What the results show.**

- **The spread follows the error.** The committee spread and the true force error are positively correlated (r ≈ 0.47 per atom), and in the binned plot the mean error rises steadily, from about 0.21 to 0.57 eV/Å, as the spread grows. The spread is a useful warning sign for atoms whose forces are less reliable.
- **But it underestimates the size of the error.** The spread (about 0.003–0.03 eV/Å) is more than ten times smaller than the true error. The two are defined a little differently (the spread is the standard deviation of one force component, the error is the length of the error vector, which makes it up to √3 larger), but the gap remains more than tenfold. All three models start from the same pretrained weights and use the same settings, so they make similar mistakes, and the spread shows only the part where they disagree.
- **By element.** P has the largest spread and error (about 0.52 eV/Å), then S and Cl (about 0.4–0.46 eV/Å), and Li the smallest (about 0.23 eV/Å). P sits in rigid PS₄ units with large, stiff P–S bond forces, and P and Cl are the least common atoms (32 of 416 each), so the model has seen fewer of their environments. Li moves on a soft energy surface with small forces: being mobile does not mean being hard to predict.
- **Active learning.** Run MD with the current model and compute the committee spread for each snapshot. Pick the 10 snapshots with the largest spread, spread out in time so that they are not near copies, compute them with DFT, add them to the training set and fine-tune again. Repeat until the spread stays small.

*Your numbers may differ slightly from run to run.*